In [ ]:

!pip install vaderSentiment textblob seaborn --quiet

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
import pickle

print("Libraries loaded successfully!")

os.makedirs("model", exist_ok=True)
os.makedirs("results", exist_ok=True)


print("Please upload IMDB Dataset.csv")
uploaded = files.upload()

df = pd.read_csv("IMDB Dataset.csv")
print("Dataset loaded!")


df['review'] = df['review'].str.lower().str.strip()
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})


analyzer = SentimentIntensityAnalyzer()

def extract_features(text):
    vader_scores = analyzer.polarity_scores(text)
    textblob_score = TextBlob(text).sentiment.polarity
    return [
        vader_scores['compound'],
        vader_scores['pos'],
        vader_scores['neg'],
        vader_scores['neu'],
        textblob_score
    ]

print("Extracting features (this may take a few minutes)...")
X = np.array([extract_features(review) for review in df['review']])
y = df['sentiment'].values


scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


model = Sequential([
    Dense(32, activation='relu', input_shape=(5,)),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)


loss, accuracy = model.evaluate(X_test, y_test)
print("\nTest Accuracy:", accuracy)

y_pred = (model.predict(X_test) > 0.5).astype(int)

report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nClassification Report:\n", report)
print("Confusion Matrix:\n", cm)


#Save Confusion Matrix Image

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("results/confusion_matrix.png")
plt.close()


#Save Loss Curves Image
plt.figure(figsize=(6,5))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train", "Validation"])
plt.tight_layout()
plt.savefig("results/loss_curves.png")
plt.close()


#Save Metrics File
with open("results/metrics.txt", "w") as f:
    f.write("IMDB Sentiment Analysis using MLP\n")
    f.write("=================================\n\n")
    f.write(f"Test Accuracy: {accuracy}\n\n")
    f.write("Classification Report:\n")
    f.write(report)
    f.write("\nConfusion Matrix:\n")
    f.write(str(cm))


#Save Results Discussion
with open("results/results_discussion.txt", "w") as f:
    f.write("Results Discussion\n")
    f.write("==================\n\n")
    f.write(f"The MLP model achieved a test accuracy of {round(accuracy,2)}.\n\n")
    f.write("Precision and recall are balanced across classes.\n")
    f.write("Loss curves show steady convergence without overfitting.\n")
    f.write("Performance is limited because only sentence-level lexicon features were used.\n")
    f.write("More advanced representations such as TF-IDF or embeddings could improve performance.\n")


with open("model/best_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("\nAll required files saved successfully!")


#Zip Results Folder 
!zip -r results_files.zip results > /dev/null
!zip -r model_files.zip model > /dev/null


#Download Files
files.download("results_files.zip")
files.download("model_files.zip")

print("\nEverything completed successfully!")


Libraries loaded successfully!
Please upload IMDB Dataset.csv


Saving IMDB Dataset.csv to IMDB Dataset (1).csv
Dataset loaded!
Extracting features (this may take a few minutes)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 32)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Everything completed successfully!
